# Comprehensive Time-Series Diagnostics (Config-Driven)

This notebook is the econometric foundation of the project. It links observable time-series properties of PHP-centered FX returns to model architecture choices.

Core outputs:
- Stationarity panel (ADF, KPSS, Phillips-Perron)
- Structural breaks and regime shifts
- Cointegration and VAR justification diagnostics
- Granger causality and cross-pair dynamics
- Rolling risk and dependency diagnostics
- Economic interpretation linked to Philippine FX fundamentals

All outputs are saved to `results/{active_target}/diagnostics/`.

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')

from statistical_validation import (
    set_reproducible_seed,
    load_config,
    get_project_paths,
    get_returns_frame,
    run_stationarity_panel,
    structural_breaks_panel,
    johansen_cointegration_table,
    select_var_lag_aic,
    granger_causality_matrix,
    acf_pacf_sqacf_table,
    rolling_diagnostics,
    cross_correlation_lead_lag,
    rolling_correlation_table,
    rolling_parameter_stability,
    garch_residual_diagnostics,
    var_irf_table,
    event_annotation_frame,
    save_table,
)

set_reproducible_seed(42)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 300)

#### Interpretation
This setup cell imports all libraries and helper functions used in diagnostics. A successful run means the analysis environment is consistent and later results are comparable across cells.

In [2]:
config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
paths = get_project_paths(config)

diagnostics_dir = paths['results_dir'] / 'diagnostics'
diagnostics_dir.mkdir(parents=True, exist_ok=True)

ret_df = get_returns_frame(config, split='fx_aligned')
ret_cols = ret_df.columns.tolist()

print('Active target:', active_target)
print('Return columns:', ret_cols)
print('Sample shape:', ret_df.shape)
print('Diagnostics output dir:', diagnostics_dir)

Active target: PHP
Return columns: ['USDPHP_RET', 'CNYPHP_RET', 'JPYPHP_RET', 'HKDPHP_RET', 'SGDPHP_RET']
Sample shape: (4230, 5)
Diagnostics output dir: results\PHP\diagnostics


#### Interpretation
This cell confirms active target, data availability, and dimensions. Use these printed values as a data quality check; mismatched target or unexpected row count can invalidate all downstream inference.

## Economic Context (Philippines)

Interpretation anchor for this notebook:
- USD/PHP co-moves with global USD cycles and US rate expectations.
- CNY/PHP reflects China trade demand and managed-float dynamics that can transmit to ASEAN currencies.
- JPY/PHP often captures safe-haven and carry-trade risk unwinds.
- HKD/PHP and SGD/PHP can proxy regional financial channel effects.
- Volatility spikes can align with global risk shocks (e.g., VIX jumps), policy surprises, and external funding conditions.

## 1) Stationarity Panel: ADF, KPSS, Phillips-Perron

In [3]:
stationarity_df = run_stationarity_panel(config, split='fx_aligned')
stationarity_df = stationarity_df.sort_values(['Pair', 'Regression'])
display(stationarity_df)

save_table(stationarity_df, diagnostics_dir / 'stationarity_panel.csv')
print('Saved:', diagnostics_dir / 'stationarity_panel.csv')

d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\Time Series analysis\Project_final_TS\VND-FX-Hybrid-Forecasting\src\statistical_validation.py:490: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_out = kpss(s, regression=("ct" if regression == "ct" else "c"), nlags="auto")
d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\Time Series analysis\Project_final_TS\VND-FX-Hybrid-Forecasting\src\statistical_validation.py:490: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_out = kpss(s, regression=("ct" if regression == "ct" else "c"), nlags="auto")
d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\Time Series analysis\Project_final_TS\VND-FX-Hybrid-Forecasting\src\statistical_validation.py:490: InterpolationWarning: The test statistic is outside 

,n,adf_stat,adf_pvalue,kpss_stat,kpss_pvalue,pp_stat,pp_pvalue,Pair,Regression,ADF_reject_5pct,KPSS_reject_5pct,PP_reject_5pct
2,4230,-11.491508,4.723982e-21,0.055504,0.1,NaN,NaN,CNYPHP_RET,constant,True,False,False
3,4230,-11.506579,2.623051e-18,0.037529,0.1,NaN,NaN,CNYPHP_RET,constant+trend,True,False,False
6,4230,-13.676994,1.424058e-25,0.138426,0.1,NaN,NaN,HKDPHP_RET,constant,True,False,False
7,4230,-13.722623,2.020211e-21,0.053024,0.1,NaN,NaN,HKDPHP_RET,constant+trend,True,False,False
4,4230,-51.165185,0.000000e+00,0.059242,0.1,NaN,NaN,JPYPHP_RET,constant,True,False,False
5,4230,-51.159490,0.000000e+00,0.056831,0.1,NaN,NaN,JPYPHP_RET,constant+trend,True,False,False
8,4230,-19.654153,0.000000e+00,0.158665,0.1,NaN,NaN,SGDPHP_RET,constant,True,False,False
9,4230,-19.691375,0.000000e+00,0.045252,0.1,NaN,NaN,SGDPHP_RET,constant+trend,True,False,False
0,4230,-13.825971,7.748544e-26,0.139175,0.1,NaN,NaN,USDPHP_RET,constant,True,False,False
1,4230,-13.872268,1.441047e-21,0.054475,0.1,NaN,NaN,USDPHP_RET,constant+trend,True,False,False


Saved: results\PHP\diagnostics\stationarity_panel.csv


#### Interpretation
Read the stationarity table by p-values and decision labels. If most return series are stationary, linear time-series models are suitable at level form; non-stationary series would require differencing or transformation.

### Interpretation notes

- ADF and PP rejecting the unit-root null support stationarity of returns.
- KPSS rejecting stationarity can still occur under breaks and volatility clustering, which is common in emerging-market FX.
- Mixed outcomes are informative and motivate break-sensitive and regime-aware diagnostics below.

## 2) Structural Break Detection (Ruptures if available, else sup-Chow proxy)

In [4]:
breaks_df = structural_breaks_panel(config, split='fx_aligned', max_breaks=3)
display(breaks_df.sort_values(['Pair', 'break_number']))

save_table(breaks_df, diagnostics_dir / 'structural_breaks_panel.csv')
print('Saved:', diagnostics_dir / 'structural_breaks_panel.csv')

,Pair,method,break_number,break_index,break_date,score,pvalue
1,CNYPHP_RET,sup_chow_mean,1,2766,2020-08-18,0.729383,0.393132
3,HKDPHP_RET,sup_chow_mean,1,795,2013-01-22,2.627345,0.105112
2,JPYPHP_RET,sup_chow_mean,1,1410,2015-06-03,1.185375,0.276327
4,SGDPHP_RET,sup_chow_mean,1,3111,2021-12-14,1.519695,0.217734
0,USDPHP_RET,sup_chow_mean,1,795,2013-01-22,2.656569,0.103197


Saved: results\PHP\diagnostics\structural_breaks_panel.csv


#### Interpretation
This breakpoint summary reports possible regime changes. Focus on break dates that align with macro or policy events, because these indicate parameter instability risk for models trained on the full sample.

In [5]:
fig_break = make_subplots(rows=len(ret_cols), cols=1, shared_xaxes=True, vertical_spacing=0.02,
                          subplot_titles=[f'{c} return with break markers' for c in ret_cols])

for i, c in enumerate(ret_cols, start=1):
    s = ret_df[c]
    fig_break.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines', name=c, line=dict(width=1.2)), row=i, col=1)
    marks = breaks_df[breaks_df['Pair'] == c]['break_date'].dropna().unique().tolist()
    for dt in marks:
        fig_break.add_vline(x=dt, line_color='crimson', line_dash='dot', opacity=0.6, row=i, col=1)

fig_break.update_layout(height=240 * len(ret_cols), width=1200, template='plotly_white', title='Structural breaks by pair')
fig_break.show()
fig_break.write_html(str(diagnostics_dir / 'structural_breaks_plot.html'))

#### Interpretation
The visualization shows where structural breaks concentrate through time. Dense clusters of breaks imply unstable dynamics and suggest using rolling evaluation or regime-aware model selection.

## 3) Cointegration and VAR Justification

In [6]:
johansen_df = johansen_cointegration_table(config, split='train', det_order=0, k_ar_diff=1)
lag_info = select_var_lag_aic(config, split='train', maxlags=config.get('var', {}).get('max_lags', 10))

display(johansen_df)
print('Suggested VAR lag orders:', lag_info)

save_table(johansen_df, diagnostics_dir / 'johansen_cointegration.csv')
save_table(pd.DataFrame([lag_info]), diagnostics_dir / 'var_lag_selection.csv')

d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\Time Series analysis\Project_final_TS\VND-FX-Hybrid-Forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


,rank_r,trace_stat,crit_90,crit_95,crit_99,reject_95pct
0,0,7851.012484,65.8202,69.8189,77.8202,True
1,1,5827.092107,44.4929,47.8545,54.6815,True
2,2,4177.723333,27.0669,29.7961,35.4628,True
3,3,2678.518984,13.4294,15.4943,19.9349,True
4,4,1325.595042,2.7055,3.8415,6.6349,True


Suggested VAR lag orders: {'aic_lag': 2, 'bic_lag': 1, 'hqic_lag': 1}


#### Interpretation
Johansen outputs indicate whether long-run cointegration relationships exist among currency returns. More significant ranks imply stronger multivariate long-run structure, which supports VAR-style joint modeling.

## 4) Granger Causality Network (minimum p-value over lags)

In [7]:
granger_mat = granger_causality_matrix(config, split='train', maxlag=min(5, config.get('var', {}).get('max_lags', 10)))
display(granger_mat)

save_table(granger_mat.reset_index().rename(columns={'index': 'Caused'}), diagnostics_dir / 'granger_min_pvalue_matrix.csv')

fig_granger = px.imshow(
    granger_mat,
    color_continuous_scale='RdYlGn_r',
    zmin=0,
    zmax=0.1,
    title='Granger causality matrix (lower p-value implies stronger predictive content)'
)
fig_granger.update_layout(width=900, height=750)
fig_granger.show()
fig_granger.write_html(str(diagnostics_dir / 'granger_heatmap.html'))

,USDPHP_RET,CNYPHP_RET,JPYPHP_RET,HKDPHP_RET,SGDPHP_RET
USDPHP_RET,NaN,0.053680,0.061886,2.691490e-02,2.274643e-28
CNYPHP_RET,8.670271e-12,NaN,0.009678,6.361632e-12,3.332814e-20
JPYPHP_RET,1.217654e-12,0.000025,NaN,9.536583e-14,1.490395e-23
HKDPHP_RET,2.942380e-01,0.016818,0.089009,NaN,1.375379e-25
SGDPHP_RET,9.477163e-02,0.030077,0.000033,6.499530e-02,NaN


#### Interpretation
The Granger table and figure describe directional predictability. Lower p-values and darker significance regions indicate stronger lead-lag channels between pairs, useful for feature engineering and spillover mapping.

## 5) Full ACF/PACF/Squared-ACF Structure (all pairs)

In [8]:
acf_panel = acf_pacf_sqacf_table(config, split='fx_aligned', nlags=40)
display(acf_panel.head(20))
save_table(acf_panel, diagnostics_dir / 'acf_pacf_sqacf_panel.csv')

for pair in ret_cols:
    sub = acf_panel[acf_panel['Pair'] == pair]
    if sub.empty:
        continue
    fig = make_subplots(rows=1, cols=3, subplot_titles=['ACF', 'PACF', 'Squared ACF'])
    fig.add_trace(go.Bar(x=sub['Lag'], y=sub['ACF'], marker_color='#2c3e50'), row=1, col=1)
    fig.add_trace(go.Bar(x=sub['Lag'], y=sub['PACF'], marker_color='#34495e'), row=1, col=2)
    fig.add_trace(go.Bar(x=sub['Lag'], y=sub['SqACF'], marker_color='#d35400'), row=1, col=3)
    ci = sub['sig_bound'].iloc[0]
    for col in [1, 2, 3]:
        fig.add_hline(y=ci, line_dash='dash', line_color='crimson', opacity=0.5, row=1, col=col)
        fig.add_hline(y=-ci, line_dash='dash', line_color='crimson', opacity=0.5, row=1, col=col)
    fig.update_layout(title=f'{pair}: dependency structure', template='plotly_white', width=1200, height=350)
    fig.show()
    fig.write_html(str(diagnostics_dir / f'acf_pacf_sqacf_{pair}.html'))

,Pair,Lag,ACF,PACF,SqACF,sig_bound
0,USDPHP_RET,1,-0.256644,-0.256704,0.357772,0.030136
1,USDPHP_RET,2,-0.027563,-0.100068,0.039963,0.030136
2,USDPHP_RET,3,0.044384,0.011831,0.067488,0.030136
3,USDPHP_RET,4,-0.036319,-0.025409,0.054178,0.030136
4,USDPHP_RET,5,0.029759,0.019041,0.013954,0.030136
5,USDPHP_RET,6,-0.005594,0.003450,0.024842,0.030136
6,USDPHP_RET,7,-0.011657,-0.008380,0.032432,0.030136
7,USDPHP_RET,8,0.013064,0.005826,0.019774,0.030136
8,USDPHP_RET,9,-0.031441,-0.028634,0.006438,0.030136
9,USDPHP_RET,10,0.056779,0.045381,0.011074,0.030136


#### Interpretation
ACF and PACF values diagnose linear memory in returns. Significant early lags support autoregressive terms; fast decay suggests short memory and favors parsimonious lag structures.

## 6) Rolling Diagnostics (252-day): Volatility, Autocorrelation, ARCH-LM p-value

In [9]:
rolling_df = rolling_diagnostics(config, split='fx_aligned', window=252, arch_lags=5)
display(rolling_df.head())
save_table(rolling_df, diagnostics_dir / 'rolling_diagnostics_252.csv')

fig_vol = px.line(
    rolling_df,
    x='Date',
    y='rolling_vol_252',
    color='Pair',
    title='252-day rolling volatility by pair',
    template='plotly_white'
)
fig_vol.show()
fig_vol.write_html(str(diagnostics_dir / 'rolling_volatility_252.html'))

fig_arch = px.line(
    rolling_df,
    x='Date',
    y='rolling_arch_pvalue',
    color='Pair',
    title='Rolling ARCH-LM p-values (volatility clustering persistence)',
    template='plotly_white'
)
fig_arch.add_hline(y=0.05, line_dash='dash', line_color='crimson')
fig_arch.show()
fig_arch.write_html(str(diagnostics_dir / 'rolling_arch_pvalues.html'))

,Date,Pair,rolling_vol_252,rolling_autocorr_l1,rolling_lb_pvalue_l1,rolling_arch_pvalue
0,2010-12-21,USDPHP_RET,0.554166,-0.150337,0.016560,6.845145e-10
1,2010-12-22,USDPHP_RET,0.554657,-0.152479,0.014983,6.662735e-10
2,2010-12-23,USDPHP_RET,0.555036,-0.150718,0.016195,6.017381e-10
3,2010-12-24,USDPHP_RET,0.554900,-0.151010,0.015891,5.523928e-10
4,2010-12-27,USDPHP_RET,0.554969,-0.151174,0.015797,4.711402e-10


#### Interpretation
Rolling volatility, autocorrelation, and ARCH summaries reveal time-varying risk and dependence. Persistent spikes indicate instability periods where static models can underperform.

## 7) Cross-Correlation and Lead-Lag Structure

In [10]:
lead_lag_df = cross_correlation_lead_lag(config, split='fx_aligned', max_lag=20)
lead_lag_df = lead_lag_df.sort_values('best_corr', key=lambda s: s.abs(), ascending=False)
display(lead_lag_df)

save_table(lead_lag_df, diagnostics_dir / 'lead_lag_cross_correlation.csv')

,Pair_A,Pair_B,best_lag,best_corr,interpretation
2,USDPHP_RET,HKDPHP_RET,0,0.996729,synchronous
5,CNYPHP_RET,HKDPHP_RET,0,0.853253,synchronous
0,USDPHP_RET,CNYPHP_RET,0,0.852989,synchronous
9,HKDPHP_RET,SGDPHP_RET,0,0.775768,synchronous
3,USDPHP_RET,SGDPHP_RET,0,0.769419,synchronous
6,CNYPHP_RET,SGDPHP_RET,0,0.741474,synchronous
8,JPYPHP_RET,SGDPHP_RET,0,0.635799,synchronous
7,JPYPHP_RET,HKDPHP_RET,0,0.595088,synchronous
1,USDPHP_RET,JPYPHP_RET,0,0.594985,synchronous
4,CNYPHP_RET,JPYPHP_RET,0,0.553406,synchronous


#### Interpretation
Lead-lag output quantifies which series tends to move first. Positive lead values for one currency imply potential short-horizon predictive advantage for that direction.

## 8) Volatility Spillover Proxies: Correlation Heatmaps and Rolling Correlation

In [11]:
corr_mat = ret_df.corr()
fig_corr = px.imshow(corr_mat, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                     title='Unconditional return correlation matrix')
fig_corr.update_layout(width=900, height=750)
fig_corr.show()
fig_corr.write_html(str(diagnostics_dir / 'unconditional_correlation_heatmap.html'))

rolling_corr_df = rolling_correlation_table(config, split='fx_aligned', window=126)
save_table(rolling_corr_df, diagnostics_dir / 'rolling_correlation_126.csv')

display(rolling_corr_df.head())

summary_roll_corr = (
    rolling_corr_df.groupby(['Pair_A', 'Pair_B'], as_index=False)['rolling_corr']
    .agg(['mean', 'std', 'min', 'max'])
    .reset_index()
)
display(summary_roll_corr)
save_table(summary_roll_corr, diagnostics_dir / 'rolling_correlation_summary.csv')

,Date,Pair_A,Pair_B,rolling_corr
0,2010-06-28,USDPHP_RET,CNYPHP_RET,0.996873
1,2010-06-29,USDPHP_RET,CNYPHP_RET,0.996820
2,2010-06-30,USDPHP_RET,CNYPHP_RET,0.996245
3,2010-07-01,USDPHP_RET,CNYPHP_RET,0.996246
4,2010-07-02,USDPHP_RET,CNYPHP_RET,0.996080


,index,Pair_A,Pair_B,mean,std,min,max
0,0,CNYPHP_RET,HKDPHP_RET,0.833101,0.138691,0.324917,0.996415
1,1,CNYPHP_RET,JPYPHP_RET,0.540851,0.152079,0.189455,0.871166
2,2,CNYPHP_RET,SGDPHP_RET,0.714577,0.163252,0.095452,0.958141
3,3,HKDPHP_RET,SGDPHP_RET,0.739150,0.159494,0.131735,0.959242
4,4,JPYPHP_RET,HKDPHP_RET,0.581757,0.167321,0.093281,0.876647
5,5,JPYPHP_RET,SGDPHP_RET,0.638966,0.132735,0.191285,0.886806
6,6,USDPHP_RET,CNYPHP_RET,0.831076,0.137569,0.356333,0.996873
7,7,USDPHP_RET,HKDPHP_RET,0.996247,0.003416,0.984447,0.999910
8,8,USDPHP_RET,JPYPHP_RET,0.580382,0.170097,0.083808,0.879623
9,9,USDPHP_RET,SGDPHP_RET,0.732525,0.162166,0.121803,0.960621


#### Interpretation
The correlation heatmap and rolling-correlation summary show co-movement intensity and its stability. High but unstable correlation suggests diversification benefits can disappear during stress regimes.

In [12]:
if not rolling_corr_df.empty:
    pair_example = rolling_corr_df[['Pair_A', 'Pair_B']].drop_duplicates().iloc[0]
    ex = rolling_corr_df[(rolling_corr_df['Pair_A'] == pair_example['Pair_A']) & (rolling_corr_df['Pair_B'] == pair_example['Pair_B'])]
    fig_rc = px.line(ex, x='Date', y='rolling_corr',
                     title=f"126-day rolling correlation: {pair_example['Pair_A']} vs {pair_example['Pair_B']}",
                     template='plotly_white')
    fig_rc.add_hline(y=0.0, line_dash='dash', line_color='gray')
    fig_rc.show()
    fig_rc.write_html(str(diagnostics_dir / 'example_rolling_correlation.html'))

#### Interpretation
IRF and volatility persistence outputs connect short-run shocks with medium-run adjustment speed. Slower decay and higher persistence imply longer risk memory, supporting hybrid designs that capture residual dynamics.

### Event Anchors for PHP FX Interpretation

Use the following as a narrative bridge when discussing break dates and volatility spikes:
- BSP tightening cycles and domestic inflation surprises.
- Federal Reserve hikes and USD funding pressure.
- China growth / trade surprises that can affect CNY-linked regional pricing.
- VIX-led risk-off episodes that amplify EM FX volatility.
- Remittance and BoP support periods that can cushion PHP depreciation pressure.